# Granularity Calibration — adaptive headroom + training-free density→size map

**H1 confirmed** (the prior sweep showed the optimal QASPER leaf size is document-dependent, spanning 50–400 tokens). Here we:

- **(a)** quantify the **adaptive headroom** an oracle sizing policy has over the single best fixed leaf size,
- **(b)** calibrate a **training-free density→size map** and evaluate it on a **held-out** split of documents,
- **(c)** contrast that parsimonious 2-parameter map with a **learned** multi-feature regressor.

**SBERT is used only for the sweep** (to measure evidence coverage); there is **no LLM** anywhere in this notebook. The density score is deterministic and training-free, and the analytical functions in `experiments.granularity_calibration` are pure stdlib (unit-tested offline).

In [1]:
!git clone -b ckraptor https://github.com/MissLostCodes/raptor.git
%cd raptor
!pip install -q -r requirements-colab.txt

Cloning into 'raptor'...
remote: Enumerating objects: 424, done.
remote: Counting objects: 100% (395/395), done.
remote: Compressing objects: 100% (260/260), done.
remote: Total 424 (delta 189), reused 296 (delta 121), pack-reused 29 (from 1)
Receiving objects: 100% (424/424), 1.35 MiB | 13.66 MiB/s, done.
Resolving deltas: 100% (189/189), done.
/content/raptor
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.1 MB/s eta 0:00:00


In [2]:
# Persist the sweep output to Drive so a disconnect never loses progress; rerun resumes.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = '/content/drive/MyDrive/raptor_runs'
except Exception as e:
    print('Not on Colab / Drive unavailable -> using local ./raptor_runs', e)
    RUN_DIR = 'raptor_runs'

import os
os.makedirs(RUN_DIR, exist_ok=True)
# Reuse the SAME sweep cache the GO/NO-GO notebook writes; '_50' marks the 50-doc run.
OUT = os.path.join(RUN_DIR, 'granularity_sweep_50.json')
print('RUN_DIR =', RUN_DIR)
print('OUT     =', OUT)

Mounted at /content/drive
RUN_DIR = /content/drive/MyDrive/raptor_runs
OUT     = /content/drive/MyDrive/raptor_runs/granularity_sweep_50.json


In [3]:
from experiments.datasets import get_loader

docs = get_loader('qasper').load(limit=50)
n_ans = sum(1 for d in docs for q in d.questions if q.evidence)
print(f'{len(docs)} QASPER papers, {n_ans} answerable (gold-evidence) questions total')

README.md:   0%|          | 0.00/9.64k [00:00<?, ?B/s]

qasper.py:   0%|          | 0.00/5.95k [00:00<?, ?B/s]

qasper/test/0000.parquet:   0%|          | 0.00/7.07M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

50 QASPER papers, 171 answerable (gold-evidence) questions total


In [ ]:
from experiments import granularity_sweep as gs, granularity_calibration as gc

SIZES = [50, 100, 150, 200, 300, 400]   # leaf token sizes to sweep
# SBERT-only, NO LLM. Resumable from OUT: first run can take a while (embeds every
# chunk of every paper at every size); reruns are fast (cached records are skipped).
records = gs.run_sweep(docs, SIZES, budget=2000, out_path=OUT)
print(f'{len(records)} (doc, size) records')
records[:3]

[granularity_sweep] doc 1/50 1911.10742: sweeping sizes [50, 100, 150, 200, 300, 400]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
import matplotlib.pyplot as plt

# Fig.1 + the headroom number: how much an adaptive policy could gain over the
# single best fixed leaf size.
h = gc.headroom(records)
print('HEADROOM:', h)
print(
    f"best fixed size = {h['best_fixed_size']} tok @ cov {h['best_fixed_cov']:.4f} | "
    f"oracle cov {h['oracle_cov']:.4f} | abs_gap {h['abs_gap']:.4f} "
    f"(+{h['rel_gain_pct']:.1f}% rel.)"
)

# Mean coverage vs fixed leaf size, with the per-doc oracle as a horizontal ceiling.
sizes_sorted = sorted({r['size'] for r in records if r['mean_evidence_coverage'] is not None})
fixed_curve = [gc.mean_coverage_at_size(records, s) for s in sizes_sorted]

plt.figure(figsize=(7, 4))
plt.plot(sizes_sorted, fixed_curve, marker='o', label='mean coverage @ fixed size')
plt.axhline(h['oracle_cov'], color='C3', ls='--',
            label=f"per-doc oracle = {h['oracle_cov']:.3f}")
plt.scatter([h['best_fixed_size']], [h['best_fixed_cov']], color='C1', zorder=5,
            label=f"best fixed = {h['best_fixed_size']} tok")
plt.xlabel('leaf chunk size (tokens)')
plt.ylabel('mean evidence coverage')
plt.title('Fig.1 — adaptive headroom: best fixed size vs per-doc oracle')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.show()

In [ ]:
from raptor.chunking.density_score import density_score

# Deterministic, training-free density signal per document (rho + its 6 features).
feats = {d.doc_id: density_score(d.text)[1] for d in docs}
rho = {d.doc_id: density_score(d.text)[0] for d in docs}

# Per-doc optimal leaf size from the sweep (the calibration target).
opt = gs.best_size_per_doc(records)
print(f'{len(opt)} docs with a defined optimal size')
from collections import Counter
print('distribution of optimal sizes:', dict(Counter(opt.values())))

In [ ]:
# Which density signal predicts granularity? Spearman rank-corr of rho and each of
# the 6 density features against the per-doc optimal leaf size (feature ablation).
doc_ids = [k for k in opt if k in rho]   # docs with both a target and a score

FEATURE_NAMES = [
    'mean_sentence_len_tokens_norm',
    'type_token_ratio',
    'numeral_symbol_density',
    'mean_word_len_chars_norm',
    'list_marker_density',
    'nonstopword_ratio',
]

opt_vec = [opt[k] for k in doc_ids]
print(f"{'signal':<32}{'spearman vs opt size':>22}")
print('-' * 54)
rc_rho = gc.rank_corr([rho[k] for k in doc_ids], opt_vec)
print(f"{'rho (combined density)':<32}{rc_rho:>22.3f}")
for f in FEATURE_NAMES:
    rc = gc.rank_corr([feats[k][f] for k in doc_ids], opt_vec)
    print(f'{f:<32}{rc:>22.3f}')

In [ ]:
# Paper Table 1: held-out evaluation of the training-free density->size map.
# Two training-free variants are compared:
#   (i)  composite rho  -> size   (the equal-weight density score)
#   (ii) BEST single feature -> size, the feature selected by |Spearman| on TRAIN
# The pilot showed the equal-weight composite CANCELS signal (features correlate
# with optimal size in opposite directions), so the selected-feature map is the
# real training-free arm; rho is kept only to show the cancellation explicitly.
tr, te = gc.train_test_split_docs(list(opt))
print(f'{len(tr)} train docs, {len(te)} test docs')

# (i) parsimonious size = a + b*rho on TRAIN only.
fit_rho = gc.fit_rho_to_size({k: rho[k] for k in tr}, {k: opt[k] for k in tr})
pred_rho = {k: gc.predict_size(rho[k], fit_rho, l_min=50, l_max=400) for k in te}

# (ii) select the strongest single density feature on TRAIN, then fit size = a + b*feature.
sel = gc.select_and_fit(feats, opt, tr, feature_names=FEATURE_NAMES)
print(f"selected feature: {sel['feature']}  (train corr {sel['corr']:+.3f})")
print('rho fit     :', fit_rho)
print('feature fit :', sel['fit'])
pred_feat = {k: gc.predict_size(feats[k][sel['feature']], sel['fit'], 50, 400) for k in te}

# Score each policy on the TEST docs (coverage snapped to the swept grid).
best_fixed_size, _ = gc.best_global_fixed(records)
cov_fixed    = gc.coverage_under_sizes(records, {k: best_fixed_size for k in te})
cov_rho      = gc.coverage_under_sizes(records, pred_rho)
cov_density  = gc.coverage_under_sizes(records, pred_feat)   # headline training-free arm
_, oracle_sizes = gc.oracle_per_doc(records)
cov_oracle   = gc.coverage_under_sizes(records, {k: oracle_sizes[k] for k in te if k in oracle_sizes})

print()
print(f"{'policy (TEST docs)':<38}{'mean coverage':>14}")
print('-' * 52)
print(f"{'best global fixed (' + str(best_fixed_size) + ' tok)':<38}{cov_fixed:>14.4f}")
print(f"{'training-free: composite rho->size':<38}{cov_rho:>14.4f}")
print(f"{'training-free: ' + str(sel['feature']) + '->size':<38}{cov_density:>14.4f}")
print(f"{'per-doc oracle (ceiling)':<38}{cov_oracle:>14.4f}")

In [ ]:
# Learned baseline: a 6-feature linear regressor (sklearn) fit on TRAIN, evaluated
# on TEST the same way. Shows whether the training-free 2-param map is competitive
# with a fully learned multi-feature model.
from sklearn.linear_model import LinearRegression

X_tr = [[feats[k][f] for f in FEATURE_NAMES] for k in tr]
y_tr = [opt[k] for k in tr]
reg = LinearRegression().fit(X_tr, y_tr)

def _clip_size(v, lo=50, hi=400):
    return int(max(lo, min(hi, round(v))))

pred_test_learned = {
    k: _clip_size(reg.predict([[feats[k][f] for f in FEATURE_NAMES]])[0]) for k in te
}
cov_learned = gc.coverage_under_sizes(records, pred_test_learned)

print(f"{'policy (TEST docs)':<40}{'mean coverage':>14}")
print('-' * 54)
print(f"{'best global fixed':<40}{cov_fixed:>14.4f}")
print(f"{'training-free: composite rho':<40}{cov_rho:>14.4f}")
print(f"{'training-free: selected feature (1 param)':<40}{cov_density:>14.4f}")
print(f"{'learned regressor (sklearn, 6 feat)':<40}{cov_learned:>14.4f}")
print(f"{'per-doc oracle (ceiling)':<40}{cov_oracle:>14.4f}")

In [ ]:
# Robustness of Table 1: repeat the held-out evaluation over many random train/test
# splits. A single 70/30 split on ~50 docs leaves a ~15-doc test set (high variance),
# so the honest H2 statistic is the DISTRIBUTION of test coverage across splits plus
# how often the training-free selected-feature map actually beats the best fixed size.
# Free: runs on the already-cached sweep records. best-global-fixed is chosen over ALL
# docs (a deliberately STRONG, mildly optimistic baseline -> a conservative H2 test).
import statistics

N_SPLITS = 50
_rows = {'best global fixed': [], 'training-free: composite rho': [],
         'training-free: selected feature': [], 'learned (6 feat)': [], 'per-doc oracle': []}
_wins_feat = 0
_sel_freq = Counter()
bf_size, _ = gc.best_global_fixed(records)
_, oracle_sizes_all = gc.oracle_per_doc(records)

for _seed in range(N_SPLITS):
    _tr, _te = gc.train_test_split_docs(list(opt), seed=_seed)
    if not _tr or not _te:
        continue
    # training-free: composite rho
    _fr = gc.fit_rho_to_size({k: rho[k] for k in _tr}, {k: opt[k] for k in _tr})
    _pr = {k: gc.predict_size(rho[k], _fr, 50, 400) for k in _te}
    # training-free: strongest single feature, selected on THIS split's train only
    _se = gc.select_and_fit(feats, opt, _tr, feature_names=FEATURE_NAMES)
    _sel_freq[_se['feature']] += 1
    _pf = ({k: gc.predict_size(feats[k][_se['feature']], _se['fit'], 50, 400) for k in _te}
           if _se['feature'] else {})
    # learned 6-feature regressor (same recipe as the cell above)
    _reg = LinearRegression().fit([[feats[k][f] for f in FEATURE_NAMES] for k in _tr],
                                  [opt[k] for k in _tr])
    _pl = {k: _clip_size(_reg.predict([[feats[k][f] for f in FEATURE_NAMES]])[0]) for k in _te}
    # score every policy on THIS split's TEST docs (snapped to the swept grid)
    _cf = gc.coverage_under_sizes(records, {k: bf_size for k in _te})
    _cr = gc.coverage_under_sizes(records, _pr)
    _cd = gc.coverage_under_sizes(records, _pf) if _pf else float('nan')
    _cl = gc.coverage_under_sizes(records, _pl)
    _co = gc.coverage_under_sizes(records,
                                  {k: oracle_sizes_all[k] for k in _te if k in oracle_sizes_all})
    _rows['best global fixed'].append(_cf)
    _rows['training-free: composite rho'].append(_cr)
    _rows['training-free: selected feature'].append(_cd)
    _rows['learned (6 feat)'].append(_cl)
    _rows['per-doc oracle'].append(_co)
    if _cd == _cd and _cd > _cf:   # _cd == _cd skips NaN splits
        _wins_feat += 1

def _mean_std(xs):
    xs = [x for x in xs if x == x]
    return (statistics.mean(xs), statistics.pstdev(xs)) if xs else (float('nan'), 0.0)

print(f'H2 robustness over {N_SPLITS} random 70/30 splits (mean +/- std TEST coverage)')
print(f"{'policy':<36}{'mean':>9}{'std':>9}")
print('-' * 54)
for _k in ['best global fixed', 'training-free: composite rho',
           'training-free: selected feature', 'learned (6 feat)', 'per-doc oracle']:
    _m, _sd = _mean_std(_rows[_k])
    print(f'{_k:<36}{_m:>9.4f}{_sd:>9.4f}')
print()
print(f'selected-feature map beats best-global-fixed in {_wins_feat}/{N_SPLITS} splits')
print('selected-feature frequency across splits:', dict(_sel_freq))

## How to read this

- **Headroom (Fig.1).** If the per-doc oracle sits well above the best fixed-size point, there is real coverage to be won by sizing leaves per document. A negligible `abs_gap` means a fixed size is already near-optimal and the adaptive story is weak.
- **Correlation (the key diagnostic).** The signal(s) with the largest-magnitude Spearman vs optimal size carry granularity information; near-zero rows are dead features. **The 8-doc pilot found the equal-weight composite `rho` is near-zero (+0.04) while individual features are strong but OPPOSITELY signed** (`mean_word_len` +0.79, `nonstopword_ratio` +0.66, `type_token_ratio` −0.60) — so averaging them cancels the signal. The composite's `invert=True` 'denser⇒smaller' assumption is also only half-right: two 'harder-text' features want *larger* leaves. That is why the headline training-free arm **selects the strongest single feature on TRAIN** rather than trusting the hand-weighted composite.
- **Adaptive wins** when, on the **held-out TEST** docs, the **selected-feature** coverage **exceeds best global fixed** and **approaches the per-doc oracle** (the composite-`rho` row is kept only to show the cancellation).
- **Training-free ≈ learned** when the parsimonious 1-parameter selected-feature map lands close to the 6-feature sklearn regressor on TEST. If so, the cheap, interpretable, label-free map is the one to ship — no per-corpus training at inference.
- **Robustness (the H2 number to quote).** The single 70/30 split above is illustrative; the repeated-split cell is the headline. Report **mean ± std TEST coverage over 50 random splits** and the **win-rate** of the selected-feature map over best-global-fixed. A clear H2 pass = a high win-rate **and** a *stable* selected feature (one feature chosen in most splits). A near-50% win-rate, large std, or a feature that flips across splits means the signal is too weak at this N — scale `limit=` before claiming H2.
- **N caveat.** Selecting one of six features on a small sample is mildly optimistic; the held-out TEST split guards against it, but treat single-feature wins at small `limit=` as directional. Scale `limit=` (the sweep caches + resumes) before quoting the correlation or the ship/no-ship call.

Scale up `limit=` if the picture is borderline; the sweep is cached in `OUT` and resumes, so re-running only embeds the newly added documents.